In [38]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [40]:
production_lines = pd.read_excel(r"C:\Users\PCexpress\Downloads\production_lines_raw.xlsx")
machines = pd.read_excel(r"C:\Users\PCexpress\Downloads\machines_raw.xlsx")
machine_production = pd.read_excel(r"C:\Users\PCexpress\Downloads\machine_production_raw.xlsx")
shifts = pd.read_csv(r"C:\Users\PCexpress\Downloads\shifts_raw.csv")
downtime_events = pd.read_csv(r"C:\Users\PCexpress\Downloads\downtime_events_raw.csv")
calendar = pd.read_csv(r"C:\Users\PCexpress\Downloads\calendar_raw.csv")
quality_inspections = pd.read_csv(r"C:\Users\PCexpress\Downloads\quality_inspections_raw.csv")
downtime_reasons = pd.read_csv(r"C:\Users\PCexpress\Downloads\downtime_reasons_raw.csv")

In [78]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================

def clean_column_names(df):
    """
    Clean column names: strip spaces, convert to lowercase snake_case
    """
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(' ', '_', regex=False)
        .str.replace('[^a-z0-9_]', '', regex=True)
    )
    return df


def trim_string_columns(df):
    """
    Trim leading/trailing spaces from all string columns
    """
    string_cols = df.select_dtypes(include=['object']).columns
    for col in string_cols:
        df[col] = df[col].astype(str).str.strip()
    return df


def normalize_categorical(series, valid_values=None):
    """
    Normalize categorical values: lowercase, strip spaces
    Replace invalid values with 'unknown' if valid_values list provided
    """
    series = series.astype(str).str.strip().str.lower()
    if valid_values:
        series = series.apply(lambda x: x if x in valid_values else 'unknown')
    return series


# ============================================================================
# TABLE-SPECIFIC CLEANING FUNCTIONS
# ============================================================================

def clean_production_lines(production_lines):
    """
    Clean production_lines table
    """
    df = production_lines.copy()
    
    # Clean column names
    df = clean_column_names(df)
    
    # Trim string columns
    df = trim_string_columns(df)
    
    # Handle primary key: drop rows where line_id is missing
    df = df.dropna(subset=['line_id'])
    
    # Convert line_id to integer
    df['line_id'] = pd.to_numeric(df['line_id'], errors='coerce').astype('Int64')
    df = df.dropna(subset=['line_id'])
    
    # Convert foreign key plant_id to integer
    df['plant_id'] = pd.to_numeric(df['plant_id'], errors='coerce').astype('Int64')
    
    # Handle missing categorical values
    df['line_name'] = df['line_name'].fillna('unknown')
    
    # Normalize line_type
    valid_line_types = ['assembly', 'smt', 'packaging']
    df['line_type'] = normalize_categorical(df['line_type'], valid_line_types)
    
    # Normalize status
    valid_status = ['active', 'inactive']
    df['status'] = normalize_categorical(df['status'], valid_status)
    
    # Remove exact duplicates
    df = df.drop_duplicates()
    
    # Remove logical duplicates (same line_id)
    df = df.drop_duplicates(subset=['line_id'], keep='first')
    
    return df


def clean_machines(machines):
    """
    Clean machines table
    """
    df = machines.copy()
    
    # Clean column names
    df = clean_column_names(df)
    
    # Trim string columns
    df = trim_string_columns(df)
    
    # Handle primary key: drop rows where machine_id is missing
    df = df.dropna(subset=['machine_id'])
    
    # Convert machine_id to integer
    df['machine_id'] = pd.to_numeric(df['machine_id'], errors='coerce').astype('Int64')
    df = df.dropna(subset=['machine_id'])
    
    # Convert foreign key line_id to integer
    df['line_id'] = pd.to_numeric(df['line_id'], errors='coerce').astype('Int64')
    
    # Handle missing categorical values
    df['machine_name'] = df['machine_name'].fillna('unknown')
    
    # Normalize machine_type
    valid_machine_types = ['robot', 'cnc', 'tester']
    df['machine_type'] = normalize_categorical(df['machine_type'], valid_machine_types)
    
    # Convert numeric field ideal_cycle_time_sec
    df['ideal_cycle_time_sec'] = pd.to_numeric(df['ideal_cycle_time_sec'], errors='coerce')
    # Impute missing with median
    df['ideal_cycle_time_sec'] = df['ideal_cycle_time_sec'].fillna(
        df['ideal_cycle_time_sec'].median()
    )
    
    # Convert installation_date to datetime
    df['installation_date'] = pd.to_datetime(df['installation_date'], errors='coerce')
    
    # Remove exact duplicates
    df = df.drop_duplicates()
    
    # Remove logical duplicates (same machine_id)
    df = df.drop_duplicates(subset=['machine_id'], keep='first')
    
    return df


def clean_shifts(shifts):
    """
    Clean shifts table
    """
    df = shifts.copy()
    
    # Clean column names
    df = clean_column_names(df)
    
    # Trim string columns
    df = trim_string_columns(df)
    
    # Handle primary key: drop rows where shift_id is missing
    df = df.dropna(subset=['shift_id'])
    
    # Convert shift_id to integer
    df['shift_id'] = pd.to_numeric(df['shift_id'], errors='coerce').astype('Int64')
    df = df.dropna(subset=['shift_id'])
    
    # Handle missing shift_name
    df['shift_name'] = df['shift_name'].fillna('unknown')
    df['shift_name'] = df['shift_name'].astype(str).str.strip().str.lower()
    
    # Convert start_time and end_time to time format
    df['start_time'] = pd.to_datetime(df['start_time'], errors='coerce', format='%H:%M:%S').dt.time
    df['end_time'] = pd.to_datetime(df['end_time'], errors='coerce', format='%H:%M:%S').dt.time
    
    # Remove exact duplicates
    df = df.drop_duplicates()
    
    # Remove logical duplicates (same shift_id)
    df = df.drop_duplicates(subset=['shift_id'], keep='first')
    
    return df


def clean_calendar(calendar):
    """
    Clean calendar table
    """
    df = calendar.copy()
    
    # Clean column names
    df = clean_column_names(df)
    
    # Handle primary key: drop rows where date_id is missing
    df = df.dropna(subset=['date_id'])
    
    # Convert date_id to integer
    df['date_id'] = pd.to_numeric(df['date_id'], errors='coerce').astype('Int64')
    df = df.dropna(subset=['date_id'])
    
    # Convert date to datetime
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    
    # If date is missing but date_id is valid (YYYYMMDD format), reconstruct
    mask = df['date'].isna() & df['date_id'].notna()
    if mask.any():
        df.loc[mask, 'date'] = pd.to_datetime(
            df.loc[mask, 'date_id'].astype(str), 
            format='%Y%m%d', 
            errors='coerce'
        )
    
    # Convert numeric fields
    df['year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')
    df['month'] = pd.to_numeric(df['month'], errors='coerce').astype('Int64')
    df['day'] = pd.to_numeric(df['day'], errors='coerce').astype('Int64')
    df['week'] = pd.to_numeric(df['week'], errors='coerce').astype('Int64')
    
    # Remove exact duplicates
    df = df.drop_duplicates()
    
    # Remove logical duplicates (same date_id)
    df = df.drop_duplicates(subset=['date_id'], keep='first')
    
    return df


def clean_machine_production(machine_production):
    """
    Clean machine_production table
    """
    df = machine_production.copy()
    
    # Clean column names
    df = clean_column_names(df)
    
    # Handle primary key: drop rows where production_id is missing
    df = df.dropna(subset=['production_id'])
    
    # Convert production_id to integer
    df['production_id'] = pd.to_numeric(df['production_id'], errors='coerce').astype('Int64')
    df = df.dropna(subset=['production_id'])
    
    # Convert foreign keys to integer
    df['machine_id'] = pd.to_numeric(df['machine_id'], errors='coerce').astype('Int64')
    df['date_id'] = pd.to_numeric(df['date_id'], errors='coerce').astype('Int64')
    df['shift_id'] = pd.to_numeric(df['shift_id'], errors='coerce').astype('Int64')
    
    # Drop rows with missing foreign keys
    df = df.dropna(subset=['machine_id', 'date_id', 'shift_id'])
    
    # Convert numeric fields
    numeric_cols = [
        'planned_production_time_min',
        'actual_run_time_min',
        'total_units_produced',
        'good_units',
        'scrap_units'
    ]
    
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            # Impute missing numeric values with zero
            df[col] = df[col].fillna(0)
    
    # Remove exact duplicates
    df = df.drop_duplicates()
    
    # Remove logical duplicates (same machine, date, shift)
    df = df.drop_duplicates(subset=['machine_id', 'date_id', 'shift_id'], keep='first')
    
    return df


def clean_downtime_events(downtime_events):
    """
    Clean downtime_events table
    """
    df = downtime_events.copy()
    
    # Clean column names
    df = clean_column_names(df)
    
    # Handle primary key: drop rows where downtime_id is missing
    df = df.dropna(subset=['downtime_id'])
    
    # Convert downtime_id to integer
    df['downtime_id'] = pd.to_numeric(df['downtime_id'], errors='coerce').astype('Int64')
    df = df.dropna(subset=['downtime_id'])
    
    # Convert foreign keys to integer
    df['machine_id'] = pd.to_numeric(df['machine_id'], errors='coerce').astype('Int64')
    df['date_id'] = pd.to_numeric(df['date_id'], errors='coerce').astype('Int64')
    df['shift_id'] = pd.to_numeric(df['shift_id'], errors='coerce').astype('Int64')
    df['downtime_reason_id'] = pd.to_numeric(df['downtime_reason_id'], errors='coerce').astype('Int64')
    
    # Drop rows with missing foreign keys
    df = df.dropna(subset=['machine_id', 'date_id', 'shift_id'])
    
    # Convert datetime fields
    df['downtime_start'] = pd.to_datetime(df['downtime_start'], errors='coerce')
    df['downtime_end'] = pd.to_datetime(df['downtime_end'], errors='coerce')
    
    # Convert numeric field downtime_duration_min
    df['downtime_duration_min'] = pd.to_numeric(df['downtime_duration_min'], errors='coerce')
    # Impute missing with median
    df['downtime_duration_min'] = df['downtime_duration_min'].fillna(
        df['downtime_duration_min'].median()
    )
    
    # Remove exact duplicates
    df = df.drop_duplicates()
    
    return df


def clean_downtime_reasons(downtime_reasons):
    """
    Clean downtime_reasons table
    """
    df = downtime_reasons.copy()
    
    # Clean column names
    df = clean_column_names(df)
    
    # Trim string columns
    df = trim_string_columns(df)
    
    # Handle primary key: drop rows where downtime_reason_id is missing
    df = df.dropna(subset=['downtime_reason_id'])
    
    # Convert downtime_reason_id to integer
    df['downtime_reason_id'] = pd.to_numeric(df['downtime_reason_id'], errors='coerce').astype('Int64')
    df = df.dropna(subset=['downtime_reason_id'])
    
    # Normalize reason_category
    valid_categories = ['mechanical', 'electrical', 'operator', 'material', 'other']
    df['reason_category'] = normalize_categorical(df['reason_category'], valid_categories)
    
    # Handle missing reason_description
    df['reason_description'] = df['reason_description'].fillna('unknown')
    df['reason_description'] = df['reason_description'].astype(str).str.strip().str.lower()
    
    # Normalize planned_flag
    valid_planned = ['planned', 'unplanned']
    df['planned_flag'] = normalize_categorical(df['planned_flag'], valid_planned)
    
    # Remove exact duplicates
    df = df.drop_duplicates()
    
    # Remove logical duplicates (same downtime_reason_id)
    df = df.drop_duplicates(subset=['downtime_reason_id'], keep='first')
    
    return df


def clean_quality_inspections(quality_inspections):
    """
    Clean quality_inspections table
    """
    df = quality_inspections.copy()
    
    # Clean column names
    df = clean_column_names(df)
    
    # Trim string columns
    df = trim_string_columns(df)
    
    # Handle primary key: drop rows where inspection_id is missing
    df = df.dropna(subset=['inspection_id'])
    
    # Convert inspection_id to integer
    df['inspection_id'] = pd.to_numeric(df['inspection_id'], errors='coerce').astype('Int64')
    df = df.dropna(subset=['inspection_id'])
    
    # Convert foreign keys to integer
    df['machine_id'] = pd.to_numeric(df['machine_id'], errors='coerce').astype('Int64')
    df['date_id'] = pd.to_numeric(df['date_id'], errors='coerce').astype('Int64')
    df['shift_id'] = pd.to_numeric(df['shift_id'], errors='coerce').astype('Int64')
    
    # Drop rows with missing foreign keys
    df = df.dropna(subset=['machine_id', 'date_id', 'shift_id'])
    
    # Convert numeric fields
    df['inspected_units'] = pd.to_numeric(df['inspected_units'], errors='coerce')
    df['defective_units'] = pd.to_numeric(df['defective_units'], errors='coerce')
    
    # Impute missing numeric values with zero
    df['inspected_units'] = df['inspected_units'].fillna(0)
    df['defective_units'] = df['defective_units'].fillna(0)
    
    # Handle missing defect_type
    df['defect_type'] = df['defect_type'].fillna('unknown')
    df['defect_type'] = df['defect_type'].astype(str).str.strip().str.lower()
    
    # Remove exact duplicates
    df = df.drop_duplicates()
    
    return df


# ============================================================================
# MAIN CLEANING PIPELINE
# ============================================================================

def clean_all_tables(production_lines, machines, shifts, calendar, 
                     machine_production, downtime_events, 
                     downtime_reasons, quality_inspections):
    """
    Clean all manufacturing tables and return cleaned DataFrames
    
    Parameters:
    - production_lines: DataFrame
    - machines: DataFrame
    - shifts: DataFrame
    - calendar: DataFrame
    - machine_production: DataFrame
    - downtime_events: DataFrame
    - downtime_reasons: DataFrame
    - quality_inspections: DataFrame
    
    Returns:
    - Dictionary containing all cleaned DataFrames
    """
    
    print("Starting data cleaning pipeline...")
    
    # Clean each table
    production_lines_clean = clean_production_lines(production_lines)
    print(f"✓ production_lines cleaned: {production_lines.shape} → {production_lines_clean.shape}")
    
    machines_clean = clean_machines(machines)
    print(f"✓ machines cleaned: {machines.shape} → {machines_clean.shape}")
    
    shifts_clean = clean_shifts(shifts)
    print(f"✓ shifts cleaned: {shifts.shape} → {shifts_clean.shape}")
    
    calendar_clean = clean_calendar(calendar)
    print(f"✓ calendar cleaned: {calendar.shape} → {calendar_clean.shape}")
    
    machine_production_clean = clean_machine_production(machine_production)
    print(f"✓ machine_production cleaned: {machine_production.shape} → {machine_production_clean.shape}")
    
    downtime_events_clean = clean_downtime_events(downtime_events)
    print(f"✓ downtime_events cleaned: {downtime_events.shape} → {downtime_events_clean.shape}")
    
    downtime_reasons_clean = clean_downtime_reasons(downtime_reasons)
    print(f"✓ downtime_reasons cleaned: {downtime_reasons.shape} → {downtime_reasons_clean.shape}")
    
    quality_inspections_clean = clean_quality_inspections(quality_inspections)
    print(f"✓ quality_inspections cleaned: {quality_inspections.shape} → {quality_inspections_clean.shape}")
    
    print("\nData cleaning pipeline completed successfully!")
    
    return {
        'production_lines': production_lines_clean,
        'machines': machines_clean,
        'shifts': shifts_clean,
        'calendar': calendar_clean,
        'machine_production': machine_production_clean,
        'downtime_events': downtime_events_clean,
        'downtime_reasons': downtime_reasons_clean,
        'quality_inspections': quality_inspections_clean
    }


# ============================================================================
# USAGE EXAMPLE
# ============================================================================

# Execute the cleaning pipeline by passing your loaded DataFrames:
# cleaned_data = clean_all_tables(
#     production_lines, 
#     machines, 
#     shifts, 
#     calendar, 
#     machine_production, 
#     downtime_events, 
#     downtime_reasons, 
#     quality_inspections
# )

# Access cleaned tables:
# production_lines_clean = cleaned_data['production_lines']
# machines_clean = cleaned_data['machines']
# etc.

In [82]:
cleaned_data = clean_all_tables(
    production_lines, 
    machines, 
   shifts, 
   calendar, 
     machine_production, 
    downtime_events, 
    downtime_reasons, 
    quality_inspections
 )



Starting data cleaning pipeline...
✓ production_lines cleaned: (200, 5) → (188, 5)
✓ machines cleaned: (400, 6) → (384, 6)
✓ shifts cleaned: (300, 4) → (283, 4)
✓ calendar cleaned: (300, 6) → (100, 6)
✓ machine_production cleaned: (1500, 10) → (928, 10)
✓ downtime_events cleaned: (300, 8) → (292, 8)
✓ downtime_reasons cleaned: (300, 4) → (32, 4)
✓ quality_inspections cleaned: (300, 7) → (293, 7)

Data cleaning pipeline completed successfully!


In [86]:
machines = cleaned_data['machines']
production_lines = cleaned_data['production_lines']
shifts = cleaned_data['shifts']
calendar= cleaned_data['calendar']
machine_production = cleaned_data['machine_production']
downtime_events = cleaned_data['downtime_events']
downtime_reasons= cleaned_data['downtime_reasons']
quality_inspections = cleaned_data['quality_inspections']

In [92]:
machines.to_excel("D:\oee\machines.xlsx", index=False)
shifts.to_excel("D:\oee\shifts.xlsx", index=False)
calendar.to_excel("D:\oee\calendar.xlsx", index=False)
machine_production.to_excel("D:\oee\machine_production.xlsx", index=False)
downtime_events.to_excel("D:\oee\downtime_events.xlsx", index=False)
production_lines.to_excel("D:\oee\production_lines.xlsx", index=False)
downtime_reasons.to_excel("D:\oee\downtime_reasons.xlsx", index=False)
quality_inspections.to_excel("D:\oee\quality_inspections.xlsx", index=False)